# 실습 6 — Task 3. 도구 연결: Agent → MCP (Day 3 / 40분)

> 시나리오: **LLM 이 도구(함수)를 골라 쓰는 흐름**을 본 뒤, 같은 도구를
> **MCP(Model Context Protocol)** 표준으로 노출한다.
>
> LangChain 의 tool-calling agent 로 직접 구현한다.
>
> **범위 안내**: Agent(Task 1~2)는 *직접 만든다*. MCP(Task 3)는 *표준 체험* 단계로,
> 이미 작성된 서버/클라이언트를 **실행해 흐름(연결→도구목록→호출)을 눈으로 확인**하는 데 초점을 둔다.
> "같은 도구가 표준 프로토콜로 어떻게 노출되는가"를 체감하는 것이 목표 — 서버를 처음부터 짜는 것은 범위 밖.

## Task
1. 도구 2개 정의 (`search_book`, `loan_status`)
2. tool-calling Agent — LLM 이 질문에 맞는 도구를 자동 선택
3. 같은 도구를 MCP 서버(`mcp_demo/library_server.py`)로 노출 → Codex CLI 등록

## 1) 도구 정의 (`@tool`)

In [4]:
from langchain_core.tools import tool

@tool
def search_book(query: str) -> str:
    """제목 또는 저자로 도서를 검색한다."""
    return f"'{query}' 검색 결과: 해리포터, 반지의 제왕, 어린 왕자"

@tool
def loan_status(member_id: str) -> str:
    """회원의 현재 대출 현황을 조회한다."""
    return f"회원 {member_id}: 대출 2권, 연체 0권"

tools = [search_book, loan_status]

## 2) tool-calling Agent — LLM 이 도구를 고른다

In [5]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LangChain 1.x 에서는 create_agent 한 줄로 tool-calling agent 를 만든다.
# (예전의 create_tool_calling_agent + AgentExecutor 가 하나로 합쳐졌다.)
agent = create_agent(
    llm,
    tools,
    system_prompt="너는 도서관 도우미다. 필요하면 도구를 사용한다.",
)

In [6]:
# create_agent 는 LangGraph 그래프를 돌려준다.
# 입력은 messages 리스트, 출력의 마지막 메시지가 최종 답변이다.
def ask(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

print(ask("해리포터 책 있어?"))
print("---")
print(ask("회원 M021 대출 현황 알려줘"))

"해리포터" 관련 도서가 있습니다. 추가로 "반지의 제왕"과 "어린 왕자"도 검색되었습니다. 더 궁금한 점이 있으면 말씀해 주세요!
---
회원 M021의 대출 현황은 다음과 같습니다:

- 대출 중인 도서: 2권
- 연체 도서: 0권


## 3) 같은 도구를 MCP 표준으로 노출 → **직접 호출해 보기**

위 두 도구는 이미 `mcp_demo/library_server.py` 에 **FastMCP** 로 구현돼 있다.
LangChain `@tool` 과 비교해 보라 — *함수 + docstring* 구조가 똑같다.

### A. 컨테이너 안에서 완결 — 클라이언트로 직접 호출 (외부 도구 불필요)
서버를 stdio 로 띄워 MCP 표준으로 통신하는 클라이언트를 함께 제공한다.
**Codex CLI 없이** 이 한 줄로 MCP 의 전 과정(연결→도구목록→호출→리소스)을 체험한다.
```bash
python mcp_demo/library_client.py
```
> 클라이언트가 서버를 자동으로 실행하므로 서버를 따로 띄울 필요 없다.
> 이것이 Codex CLI·Cursor 같은 'MCP 호스트' 가 내부적으로 하는 일과 똑같다.

### B. (선택) 외부 호스트에 등록 — OpenAI Codex CLI
Codex CLI 가 설치돼 있다면 `~/.codex/config.json` 에 등록해 LLM 이 호출하게 할 수도 있다.
```json
{
  "mcpServers": {
    "library-tools": { "command": "python", "args": ["mcp_demo/library_server.py"] }
  }
}
```
등록 후 Codex 에게 *"M021 대출 현황 알려줘"* 라고 하면, Codex 가 MCP 도구를 호출한다.

## 회고
- [ ] LangChain tool vs MCP tool — 같은 점 / 다른 점 한 줄
- [ ] 클라이언트 출력에서 도구 `description` 이 서버 docstring 과 같은지 확인
- [ ] RAG / 웹챗봇 / MCP 중 **회사에 가장 먼저 가져갈 것** 1개 + 이유

